In [ ]:
using Flux
using Enzyme
using Random
using Statistics
using LinearAlgebra
using Pkg


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [11]:
#Here i define my "problematic" custom layer

struct G1Layer
    W_eta::Vector{Float32}
    W_Fs::Vector{Float32}
    cutoff::Float32
    charge::Float32
end

Flux.@layer G1Layer trainable = (W_eta,W_Fs,)


function G1Layer(N_G1::Int, cutoff::Float32, charge::Float32; seed::Union{Int,Nothing}=nothing)
    rng = seed === nothing ? Random.GLOBAL_RNG : MersenneTwister(seed)

    # Avoid Fs too close to zero to prevent huge contributions
    r_min = 0.1f0
    W_Fs = range(r_min, cutoff, length=N_G1) .+ 0.01f0 .* rand(rng, Float32, N_G1)

    # Compute average spacing and set eta proportional to 1/(spacing^2)
    delta = diff(W_Fs)
    avg_spacing = mean(delta)
    eta_base = 1.0f0 / (avg_spacing^2)
    W_eta = eta_base .* (0.8f0 .+ 0.4f0 .* rand(rng, Float32, N_G1))

    return G1Layer(W_eta, W_Fs, cutoff, charge)
end


function (layer::G1Layer)(x::AbstractMatrix{Float32})
    n_batch, n_neighbors = size(x)
    n_features = size(layer.W_eta, 1)

    output = zeros(Float32, n_features, n_batch)

    @inbounds for b in 1:n_batch
        for f in 1:n_features
            s = 0f0
            for n in 1:n_neighbors
                dx = x[b, n] - layer.W_Fs[f]
                s += fc(x[b, n], layer.cutoff) * exp(-layer.W_eta[f] * dx * dx)
            end
            output[f, b] = 0.1f0 * layer.charge * s
        end
    end

    return output
end

function fc(
    Rij::Float32, Rc::Float32
    ) 
    if Rij >= Rc
        return zero(Float32)
    end

    ε = eps(Float32)  
    denom = 1 - (Rij / Rc)^2
    if denom < ε
        return zero(Float32)
    end

    arg = 1 - 1 / denom
    return (exp(arg))
end

fc (generic function with 1 method)

In [12]:
#This is the loss containing a gradient

function losss(m , x , y)

    e_L = mean((m(x) .- y) .^2)
    f_L = Enzyme.gradient(Reverse , (mm,xx) -> mm(xx)[1], Const(m) , x)[2]

    return e_L + norm(f_L)
end

losss (generic function with 1 method)

In [13]:
#Test number 1, using just a native Flux layer

#definition of input and target 
x = rand(Float32 , 5)
y = sum(rand(Float32 , 5) .* x')


model = Dense(5,1)
println("The output of the model is: ",model(x))

o =  OptimiserChain(ClipNorm(1.0), Adam(0.1))
opt = Flux.setup(o, model)
grad = Enzyme.gradient(set_runtime_activity(Reverse) , (m , xx , yy)-> losss(m , xx ,yy) , model , Const(x) , Const(y))[1]


The output of the model is: Float32[1.128418]


AssertionError: AssertionError: Enzyme internal error unsupported got(arg1)
inst=LLVM.LoadInst(%sgemv_64_ = load atomic ptr, ptr @jlplt_sgemv_64__56317_got unordered, align 8, !dbg !229)
fname=sgemv_64_
FT=LLVM.FunctionType(void (ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, i32))
fn_got=LLVM.GlobalVariable("jlplt_sgemv_64__56317_got")
init=define private void @jlplt_sgemv_64__56317(ptr %0, ptr %1, ptr %2, ptr %3, ptr %4, ptr %5, ptr %6, ptr %7, ptr %8, ptr %9, ptr %10, i32 %11) #14 {
top:
  %sgemv_64_.cached = load atomic ptr, ptr @ccall_sgemv_64__56316 unordered, align 8
  %is_cached = icmp ne ptr %sgemv_64_.cached, null
  br i1 %is_cached, label %ccall, label %dlsym

dlsym:                                            ; preds = %top
  %sgemv_64_.found = call ptr @ijl_load_and_lookup(ptr @_j_str_libblastrampoline_5_dll_5, ptr @_j_str_sgemv_64__9, ptr @ccalllib_libblastrampoline_5_dll56214)
  store atomic ptr %sgemv_64_.found, ptr @ccall_sgemv_64__56316 release, align 8
  br la

In [14]:
#Test number 2, using just a native Flux layer inside a Chain

model = Chain(
                Dense(5,1))
println("The output of the model is: ",model(x))

o =  OptimiserChain(ClipNorm(1.0), Adam(0.1))

opt = Flux.setup(o, model)
grad = Enzyme.gradient(set_runtime_activity(Reverse) , (m , xx , yy)-> losss(m , xx ,yy) , model , Const(x) , Const(y))[1]


The output of the model is: Float32[-1.0912338]


AssertionError: AssertionError: Enzyme internal error unsupported got(arg1)
inst=LLVM.LoadInst(%sgemv_64_ = load atomic ptr, ptr @jlplt_sgemv_64__59354_got unordered, align 8, !dbg !229)
fname=sgemv_64_
FT=LLVM.FunctionType(void (ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, i32))
fn_got=LLVM.GlobalVariable("jlplt_sgemv_64__59354_got")
init=define private void @jlplt_sgemv_64__59354(ptr %0, ptr %1, ptr %2, ptr %3, ptr %4, ptr %5, ptr %6, ptr %7, ptr %8, ptr %9, ptr %10, i32 %11) #14 {
top:
  %sgemv_64_.cached = load atomic ptr, ptr @ccall_sgemv_64__59353 unordered, align 8
  %is_cached = icmp ne ptr %sgemv_64_.cached, null
  br i1 %is_cached, label %ccall, label %dlsym

dlsym:                                            ; preds = %top
  %sgemv_64_.found = call ptr @ijl_load_and_lookup(ptr @_j_str_libblastrampoline_5_dll_5, ptr @_j_str_sgemv_64__9, ptr @ccalllib_libblastrampoline_5_dll59251)
  store atomic ptr %sgemv_64_.found, ptr @ccall_sgemv_64__59353 release, align 8
  br la

In [15]:
#Test number 3, using my custom layer

#definition of input and target 
x = rand(Float32 , 1 , 5)

y = sum(rand(Float32 , 5) .* x')
model = G1Layer(2 , 5.0f0 , 2.0f0)
println("The output of the model is: ",model(x))

o =  OptimiserChain(ClipNorm(1.0), Adam(0.1))

opt = Flux.setup(o, model)
grad = Enzyme.gradient(set_runtime_activity(Reverse) , (m , xx , yy)-> losss(m , xx ,yy) , model , Const(x) , Const(y))[1]


The output of the model is: Float32[0.98813784; 0.42700633;;]


AssertionError: AssertionError: Enzyme internal error unsupported got(arg1)
inst=LLVM.LoadInst(%snrm2_64_ = load atomic ptr, ptr @jlplt_snrm2_64__60403_got unordered, align 8, !dbg !123)
fname=snrm2_64_
FT=LLVM.FunctionType(float (ptr, ptr, ptr))
fn_got=LLVM.GlobalVariable("jlplt_snrm2_64__60403_got")
init=define private float @jlplt_snrm2_64__60403(ptr %0, ptr %1, ptr %2) #5 {
top:
  %snrm2_64_.cached = load atomic ptr, ptr @ccall_snrm2_64__60402 unordered, align 8
  %is_cached = icmp ne ptr %snrm2_64_.cached, null
  br i1 %is_cached, label %ccall, label %dlsym

dlsym:                                            ; preds = %top
  %snrm2_64_.found = call ptr @ijl_load_and_lookup(ptr @_j_str_libblastrampoline_5_dll_5, ptr @_j_str_snrm2_64__4, ptr @ccalllib_libblastrampoline_5_dll60401)
  store atomic ptr %snrm2_64_.found, ptr @ccall_snrm2_64__60402 release, align 8
  br label %ccall

ccall:                                            ; preds = %dlsym, %top
  %snrm2_64_ = phi ptr [ %snrm2_6

In [16]:
#Test number 4, using my custom layer inside a Chain

model = Chain(
            G1Layer(2 , 5.0f0 , 2.0f0))
println("The output of the model is: ",model(x))

o =  OptimiserChain(ClipNorm(1.0), Adam(0.1))

opt = Flux.setup(o, model)
grad = Enzyme.gradient(set_runtime_activity(Reverse) , (m , xx , yy)-> losss(m , xx ,yy) , model , Const(x) , Const(y))[1]


The output of the model is: Float32[0.98896646; 0.3485074;;]


AssertionError: AssertionError: Enzyme internal error unsupported got(arg1)
inst=LLVM.LoadInst(%snrm2_64_ = load atomic ptr, ptr @jlplt_snrm2_64__61323_got unordered, align 8, !dbg !123)
fname=snrm2_64_
FT=LLVM.FunctionType(float (ptr, ptr, ptr))
fn_got=LLVM.GlobalVariable("jlplt_snrm2_64__61323_got")
init=define private float @jlplt_snrm2_64__61323(ptr %0, ptr %1, ptr %2) #5 {
top:
  %snrm2_64_.cached = load atomic ptr, ptr @ccall_snrm2_64__61322 unordered, align 8
  %is_cached = icmp ne ptr %snrm2_64_.cached, null
  br i1 %is_cached, label %ccall, label %dlsym

dlsym:                                            ; preds = %top
  %snrm2_64_.found = call ptr @ijl_load_and_lookup(ptr @_j_str_libblastrampoline_5_dll_5, ptr @_j_str_snrm2_64__4, ptr @ccalllib_libblastrampoline_5_dll61321)
  store atomic ptr %snrm2_64_.found, ptr @ccall_snrm2_64__61322 release, align 8
  br label %ccall

ccall:                                            ; preds = %dlsym, %top
  %snrm2_64_ = phi ptr [ %snrm2_6

In [17]:
#Test number 4, using my custom layer chained to a Dense layer

model = Chain(
            G1Layer(2 , 5.0f0 , 2.0f0),
            Dense(2 , 1))
println("The output of the model is: ",model(x))

o =  OptimiserChain(ClipNorm(1.0), Adam(0.1))

opt = Flux.setup(o, model)
grad = Enzyme.gradient(set_runtime_activity(Reverse) , (m , xx , yy)-> losss(m , xx ,yy) , model , Const(x) , Const(y))[1]

Flux.update!(opt, model, grad)

The output of the model is: Float32[-1.0690206;;]


AssertionError: AssertionError: Enzyme internal error unsupported got(arg1)
inst=LLVM.LoadInst(%sgemm_64_ = load atomic ptr, ptr @jlplt_sgemm_64__65599_got unordered, align 8, !dbg !184)
fname=sgemm_64_
FT=LLVM.FunctionType(void (ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, ptr, i32, i32))
fn_got=LLVM.GlobalVariable("jlplt_sgemm_64__65599_got")
init=define private void @jlplt_sgemm_64__65599(ptr %0, ptr %1, ptr %2, ptr %3, ptr %4, ptr %5, ptr %6, ptr %7, ptr %8, ptr %9, ptr %10, ptr %11, ptr %12, i32 %13, i32 %14) #12 {
top:
  %sgemm_64_.cached = load atomic ptr, ptr @ccall_sgemm_64__65598 unordered, align 8
  %is_cached = icmp ne ptr %sgemm_64_.cached, null
  br i1 %is_cached, label %ccall, label %dlsym

dlsym:                                            ; preds = %top
  %sgemm_64_.found = call ptr @ijl_load_and_lookup(ptr @_j_str_libblastrampoline_5_dll_5, ptr @_j_str_sgemm_64__9, ptr @ccalllib_libblastrampoline_5_dll65436)
  store atomic ptr %sgemm_64_.found, ptr @ccal